#### This code is pulling public-sector contract award data from Contracts Finder, finding the Companies House number of winning suppliers, summarising contract wins per company

In [1]:
import time
import requests
import pandas as pd
from pathlib import Path

CF_BASE = "https://www.contractsfinder.service.gov.uk"

output_dir = Path("C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\notebooks")


In [2]:
# Pull award notices from contract finder
def cf_search(published_from, published_to, stages="award", page=1, size=None):
    """
    Pull one batch of Contracts Finder OCDS notices.
    """

    url = f"{CF_BASE}/Published/Notices/OCDS/Search"

    params = {
        "publishedFrom": published_from,
        "publishedTo": published_to,
        "stages": stages,
        "order": "DESC",
        "size":size,
        "page":page
    }

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Pulling multiple batches
def cf_search_all(published_from, published_to, stages="award", limit=100, max_batches=30):
    "Pulling multiple batches of award notices"
    all_releases = []
    page=1
    while page <= max_batches:
        try:
            data = cf_search(published_from, published_to, stages=stages, page=page,size=limit)
        except:
            status =getattr(e.response, "status_code", None)
            if status ==429:
                print(f"Rate limited on page {page} - waiting 120s then retrying same page ")
                time.sleep(120)
                continue
            raise

        releases = data.get("releases", [])
        if not releases:
            break
        all_releases.extend(releases)
        page+=1
        time.sleep(0.5)
    return all_releases


In [4]:
#taking one OCDS release and turning it into multiple rows
def flatten_award_release(release):
    rows = []
    buyer_name = (release.get("buyer") or {}).get("name")
    tender_title = (release.get("tender") or {}).get("title")
    ocid = release.get("ocid")
    # Mapping each supplier party to its Companies House number 
    company_number_by_party_id = {}
    for party in release.get("parties", []):
        identifier = party.get("identifier") or {}
        if "supplier" in party.get("roles", []) and identifier.get("scheme") == "GB-COH":
            company_number_by_party_id[party.get("id")] = str(identifier.get("id")).zfill(8)

    # One row per supplier on each award
    for award in release.get("awards", []):
        award_date = award.get("date")
        award_value = (award.get("value") or {}).get("amount")
        for supplier in award.get("suppliers", []):         
            company_number = company_number_by_party_id.get(supplier.get("id"))
            if company_number is None:
                continue                                  
            rows.append({                                
                "CompanyNumber": company_number,
                "supplier_name_cf": supplier.get("name"),
                "buyer_name": buyer_name,
                "contract_title": tender_title,
                "award_date": award_date,
                "award_value_gbp": award_value,
                "ocid": ocid,
            })
    return rows

In [7]:
# creating contract finder dataset
award_releases = cf_search_all(
    published_from="2025-01-01",
    published_to="2026-06-27",
    stages="award",
    limit=100,
    max_batches=10
)
print("releases returned", len(award_releases))

raw_rows = []
for release in award_releases:
    raw_rows.extend(flatten_award_release(release))

print("raw_rows built:", len(raw_rows))                          

expected_cols = [                                                  
    "CompanyNumber", "supplier_name_cf", "award_date",
    "award_value_gbp", "buyer_name", "contract_title", "ocid",
]

contracts_finder_awards_df = pd.DataFrame(raw_rows, columns=expected_cols)

contracts_finder_awards_df.head()

releases returned 1000
raw_rows built: 330


,CompanyNumber,supplier_name_cf,award_date,award_value_gbp,buyer_name,contract_title,ocid
0,SC115530,RSK Environment Ltd,2026-06-20T00:00:00+01:00,82907.94,NHS PROPERTY SERVICES LIMITED,Consultancy for Phase Two Hayman & Lanyon Stru...,ocds-b5fd17-1330ea2e-22ea-4844-81d0-fe3021c678a2
1,03794455,Terberg Matec UK - a trading division of Denni...,2026-06-19T00:00:00+01:00,1115360.40,Bassetlaw District Council,Food waste vehicle,ocds-b5fd17-a48f6e01-9606-4532-a828-791816e414bb
2,01800000,BRITISH TELECOMMUNICATIONS PLC,2026-06-24T00:00:00+01:00,251352.70,H M REVENUE & CUSTOMS,WAN Circuit for Regional Centre,ocds-b5fd17-5998e41f-f87b-4d3f-b17b-91e1d855a3bc
3,06903140,Reed Specialist Recruitment Limited,2022-05-13T00:00:00+01:00,226839.37,UK SHARED BUSINESS SERVICES LIMITED,CS22282 RM6160 Administrator,ocds-b5fd17-eb8ccd91-f259-4ffd-a9a0-ef0266fa99a5
4,04655948,Media Zoo Limited,2026-06-08T00:00:00+01:00,48640.00,Defence Science and Technology Laboratory,Experts Mediazoo,ocds-b5fd17-8ee9309e-122b-422f-a017-833f5a1211ed


In [8]:
contracts_finder_awards_df["CompanyNumber"] = (
    contracts_finder_awards_df["CompanyNumber"]
    .astype(str)
    .str.zfill(8)
)

contracts_finder_awards_df["award_date"] = pd.to_datetime(
    contracts_finder_awards_df["award_date"],
    errors="coerce",
)

contracts_finder_awards_df["award_value_gbp"] = pd.to_numeric(
    contracts_finder_awards_df["award_value_gbp"],
    errors="coerce",
)

C:\conda_temp\ipykernel_2496\2378339436.py:7: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  contracts_finder_awards_df["award_date"] = pd.to_datetime(


In [9]:
# Summarise to one row per company, sort by total contract value (most useful for an RM)
contracts_won = (
    contracts_finder_awards_df.groupby("CompanyNumber", as_index=False)
    .agg(
        supplier_name=("supplier_name_cf", "first"),
        public_contracts_won=("ocid", "nunique"),
        latest_award_date=("award_date", "max"),
        total_award_value_gbp=("award_value_gbp", "sum"),
    ).sort_values("total_award_value_gbp", ascending=False).reset_index(drop=True)
)

print("Companies with at least one public contract:", len(contracts_won))
contracts_won.head(20)

Companies with at least one public contract: 25


,CompanyNumber,supplier_name,public_contracts_won,latest_award_date,total_award_value_gbp
0,02174990,SoftCat Plc,3,2026-05-22 00:00:00+01:00,64154038.8
1,00414220,PA CONSULTING SERVICES LIMITED,1,2026-04-01 00:00:00+01:00,18291600.0
2,06263424,Your NRG,1,2026-04-28 00:00:00+01:00,15000000.0
3,03794455,Terberg Matec UK - a trading division of Denni...,1,2026-06-19 00:00:00+01:00,11153604.0
4,211199050,Made Tech Limited,1,2026-04-13 00:00:00+01:00,9051580.0
5,05907841,Opus 2 International Limited,1,2026-03-09 00:00:00+00:00,7500000.0
6,06903140,Reed Specialist Recruitment Limited,6,2026-06-23 00:00:00+01:00,7338876.3
7,02579852,Insight Direct (UK) Ltd,1,2024-03-13 00:00:00+00:00,6907379.6
8,04394343,Rand Associates Consultancy Services Ltd,1,2026-03-26 00:00:00+00:00,5189980.0
9,00641659,Blue Arrow Ltd,1,2026-06-26 00:00:00+01:00,4000000.0
